# Trabalho Prático — Aprendizagem Profunda: Deteção de Texto AI vs. Human (NumPy Puro)
Este notebook implementa um modelo de Deep Neural Network (DNN) e Regressão Logística construído **exclusivamente com NumPy**, incorporando melhorias avançadas:
- Função de ativação **ELU (Exponential Linear Unit)** para melhor fluxo de gradientes.
- Inicialização adaptativa de pesos (**He/Xavier Initialization**).
- **Learning Rate Decay** automático e **Early Stopping** para evitar overfitting.
- Otimizador **Adam** implementado do zero.

In [1]:
import numpy as np
import re, math, json, csv, os
from collections import Counter

SEED = 42
DATASET_PATH    = "AI_Human.csv"
TEACHER_INPUTS  = "dataset1_inputs.csv"
TEACHER_OUTPUTS = "dataset1_outputs.csv"
MAX_FEATURES = 5000
MAX_TRAIN, MAX_VAL, MAX_TEST = 40000, 10000, 10000
print("Ambiente configurado com sucesso!")

Ambiente configurado com sucesso!


## 1. Funções de Pré-processamento e TF-IDF

In [2]:
HUMAN_VALUES = {"0", "0.0", "false", "no", "n", "human", "humano", "real", "not generated"}
AI_VALUES    = {"1", "1.0", "true", "yes", "y", "ai", "generated", "machine", "chatgpt", "gpt"}

def parse_label(value):
    v = str(value).strip().lower()
    if v in HUMAN_VALUES: return 0
    if v in AI_VALUES:    return 1
    if "human" in v: return 0
    if any(k in v for k in ("ai", "gpt", "machine", "generated")): return 1
    raise ValueError(f"Label NÃO reconhecido: {value!r}")

def load_ai_human_csv(path):
    texts, labels = [], []
    with open(path, "r", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        cols = [c for c in (reader.fieldnames or []) if c]
        text_col = next((c for c in cols if c.strip().lower() == "text"), "text")
        lab_col  = next((c for c in cols if c.strip().lower().startswith("generated")),
                        next((c for c in cols if c.strip().lower() in {"label", "class"}), "generated"))
        for row in reader:
            text = (row.get(text_col) or "").strip()
            if not text: continue
            texts.append(text)
            labels.append(parse_label(row.get(lab_col)))
    return texts, np.array(labels, dtype=np.int64)

def split_data(texts, labels, val_frac=0.15, test_frac=0.15, seed=SEED):
    n = len(texts)
    rng = np.random.RandomState(seed)
    idx = rng.permutation(n)
    n_test, n_val = int(n * test_frac), int(n * val_frac)
    texts = np.asarray(texts, dtype=object); labels = np.asarray(labels)
    return (texts[idx[n_test+n_val:]].tolist(), texts[idx[n_test:n_test+n_val]].tolist(),
            texts[idx[:n_test]].tolist(),
            labels[idx[n_test+n_val:]], labels[idx[n_test:n_test+n_val]], labels[idx[:n_test]])

def sample_split(texts, labels, n, seed=SEED):
    if len(texts) <= n: return texts, labels
    rng = np.random.RandomState(seed)
    idx = np.sort(rng.choice(len(texts), n, replace=False))
    return [texts[i] for i in idx], labels[idx]

## 2. Vetorização TF-IDF do Zero

In [3]:
STOPWORDS = set("""a an and are as at be by for from has have he in is it its of on or
that the to was were will with you your this these those i we they them his her their
not no yes can could should would may might must shall about into over after before
between under more most very such only own same so than too also just because but if
then else when where why how all any both each few other some through during above
below again further once here there""".split())

def tokenize(text):
    text = str(text).lower()
    tokens = re.findall(r"[a-z']+", text)
    return [t.strip("'") for t in tokens if len(t) > 1 and t.strip("'") not in STOPWORDS]

def fit_text_vectorizer(texts, max_features=MAX_FEATURES):
    counts, df = Counter(), Counter()
    for text in texts:
        tokens = tokenize(text)
        counts.update(tokens); df.update(set(tokens))
    vocab = {w: i for i, (w, _) in enumerate(counts.most_common(max_features))}
    N = len(texts)
    idf = np.zeros(len(vocab), dtype=np.float32)
    for w, i in vocab.items():
        idf[i] = math.log((1 + N) / (1 + df.get(w, 0))) + 1.0
    return vocab, idf

def texts_to_tfidf_direct(texts, vocab, idf):
    X = np.zeros((len(texts), len(vocab)), dtype=np.float32)
    for i, text in enumerate(texts):
        counts = Counter(t for t in tokenize(text) if t in vocab)
        if counts:
            idx  = np.array([vocab[t] for t in counts], dtype=np.int64)
            vals = np.array([(1.0 + math.log(c)) * idf[vocab[t]]
                             for t, c in counts.items()], dtype=np.float32)
            X[i, idx] = vals
        n = np.linalg.norm(X[i])
        if n > 0: X[i] /= n
    return X

## 3. Modelo DNN com Melhorias (ELU + LR Decay + Dropout)

In [4]:
def sigmoid(z): return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))

class BinaryDNNNumpyImproved:
    def __init__(self, hidden_dims=(256, 128), activation="elu",
                 learning_rate=1e-3, lambda_l2=1e-5, dropout_rate=0.3, lr_decay=0.9, seed=SEED):
        self.hidden_dims, self.activation = list(hidden_dims), activation
        self.learning_rate, self.lambda_l2 = learning_rate, lambda_l2
        self.dropout_rate, self.lr_decay, self.seed = dropout_rate, lr_decay, seed
        self.params, self.layer_dims = {}, None

    def _init_params(self, input_dim):
        dims = [input_dim] + self.hidden_dims + [1]; self.layer_dims = dims
        np.random.seed(self.seed); self.params = {}
        for l in range(1, len(dims)):
            fi, fo = dims[l-1], dims[l]
            sc = np.sqrt(2.0 / fi) if self.activation in ("relu", "leaky_relu", "elu") else np.sqrt(1.0 / fi)
            self.params[f"W{l}"] = (np.random.randn(fo, fi) * sc).astype(np.float32)
            self.params[f"b{l}"] = np.zeros((fo, 1), dtype=np.float32)

    def _activate(self, Z, k):
        if k == "relu": return np.maximum(0, Z)
        if k == "leaky_relu": return np.where(Z > 0, Z, 0.01 * Z)
        if k == "elu": 
            alpha = 1.0
            return np.where(Z > 0, Z, alpha * (np.exp(np.clip(Z, -30, 30)) - 1))
        if k == "tanh": return np.tanh(Z)
        raise ValueError(k)

    def _activation_deriv(self, Z, k):
        if k == "relu": return (Z > 0).astype(np.float32)
        if k == "leaky_relu": return np.where(Z > 0, 1.0, 0.01).astype(np.float32)
        if k == "elu":
            alpha = 1.0
            return np.where(Z > 0, 1.0, self._activate(Z, "elu") + alpha).astype(np.float32)
        if k == "tanh": return 1.0 - np.tanh(Z) ** 2
        raise ValueError(k)

    def _forward(self, X, training=False):
        A = X.astype(np.float32); caches = []; L = len(self.layer_dims) - 1
        for l in range(1, L + 1):
            W, b = self.params[f"W{l}"], self.params[f"b{l}"]
            A_prev = A; Z = W.dot(A_prev) + b
            if l < L:
                A = self._activate(Z, self.activation); mask = None
                if training and self.dropout_rate > 0:
                    mask = (np.random.rand(*A.shape) >= self.dropout_rate).astype(np.float32) / (1.0 - self.dropout_rate)
                    A = A * mask
                caches.append((A_prev, Z, mask))
            else:
                A = sigmoid(Z); caches.append((A_prev, Z, None))
        return A, caches

    def _backward(self, Y, A, caches, m):
        L = len(self.layer_dims) - 1; grads = {}; dA = None
        for l in range(L, 0, -1):
            A_prev, Z, mask = caches[l-1]; W = self.params[f"W{l}"]
            dZ = (A - Y) if l == L else (dA * mask if mask is not None else dA) * self._activation_deriv(Z, self.activation)
            grads[f"dW{l}"] = dZ.dot(A_prev.T) / m + (self.lambda_l2 / m) * W
            grads[f"db{l}"] = np.sum(dZ, axis=1, keepdims=True) / m
            if l > 1: dA = W.T.dot(dZ)
        return grads

    def _init_adam(self):
        self.m = {k: np.zeros_like(v) for k, v in self.params.items()}
        self.v = {k: np.zeros_like(v) for k, v in self.params.items()}; self.t = 0

    def _update_adam(self, grads):
        self.t += 1; b1, b2, eps = 0.9, 0.999, 1e-8
        for k in self.params:
            g = grads["d" + k]
            self.m[k] = b1*self.m[k] + (1-b1)*g; self.v[k] = b2*self.v[k] + (1-b2)*g*g
            self.params[k] -= self.learning_rate * (self.m[k]/(1-b1**self.t)) / (np.sqrt(self.v[k]/(1-b2**self.t)) + eps)

    def loss(self, X, Y):
        A, _ = self._forward(X, training=False); eps = 1e-12; m = Y.shape[1]
        ce = -np.mean(Y*np.log(A+eps) + (1-Y)*np.log(1-A+eps))
        if self.lambda_l2 > 0:
            ce += (self.lambda_l2/(2*m)) * sum(np.sum(self.params[f"W{l}"]**2) for l in range(1, len(self.layer_dims)))
        return float(ce)

    def fit(self, X, Y, X_val=None, Y_val=None, epochs=80, batch_size=64, patience=7, verbose=True):
        np.random.seed(self.seed); self._init_params(X.shape[0]); self._init_adam()
        m = X.shape[1]; Y = Y.reshape(1, -1).astype(np.float32)
        if Y_val is not None: Y_val = Y_val.reshape(1, -1).astype(np.float32)
        best_loss, best_params, bad = np.inf, None, 0
        for epoch in range(epochs):
            idx = np.random.permutation(m)
            for s in range(0, m, batch_size):
                b = idx[s:s+batch_size]
                A, caches = self._forward(X[:, b], training=True)
                self._update_adam(self._backward(Y[:, b], A, caches, len(b)))
            tr = self.loss(X, Y); val = self.loss(X_val, Y_val) if X_val is not None else tr
            if verbose: print(f"Epoch {epoch+1:03d} | LR: {self.learning_rate:.5f} | train {tr:.5f} | val {val:.5f}")
            if val < best_loss - 1e-5:
                best_loss, best_params, bad = val, {k: v.copy() for k, v in self.params.items()}, 0
            else:
                bad += 1
                if bad >= 2: self.learning_rate *= self.lr_decay
                if bad >= patience:
                    if verbose: print("Early stopping acionado.")
                    break
        if best_params is not None: self.params = best_params
        return self

    def predict_proba(self, X):
        A, _ = self._forward(X, training=False); return A.flatten()

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(np.int64)

## 4. Treino e Avaliação do Modelo (com Métricas Completas)

In [5]:
def classification_report_numpy(y_true, y_pred):
    y_true = np.asarray(y_true).flatten().astype(int)
    y_pred = np.asarray(y_pred).flatten().astype(int)
    rep = {}
    for cls in (0, 1):
        tp = np.sum((y_true == cls) & (y_pred == cls))
        fp = np.sum((y_true != cls) & (y_pred == cls))
        fn = np.sum((y_true == cls) & (y_pred != cls))
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
        rep[cls] = {"precision": float(prec), "recall": float(rec), "f1": float(f1)}
    rep["accuracy"] = float(np.mean(y_true == y_pred))
    rep["macro_f1"] = (rep[0]["f1"] + rep[1]["f1"]) / 2
    return rep

def print_report(name, y_true, y_pred):
    rep = classification_report_numpy(y_true, y_pred)
    print(f"=== {name} ===  accuracy={rep['accuracy']:.4f} | macro_f1={rep['macro_f1']:.4f}")
    for cls, tag in ((0, "human"), (1, "ai")):
        r = rep[cls]
        print(f"  {tag:>5}: precision={r['precision']:.4f}  recall={r['recall']:.4f}  f1={r['f1']:.4f}")

# Execução do Treino e Avaliação
if os.path.exists(DATASET_PATH):
    texts, labels = load_ai_human_csv(DATASET_PATH)
    print(f"Total de exemplos: {len(texts)} | [human, ai] = {np.bincount(labels)}")
    
    texts_tr, texts_va, texts_te, y_tr, y_va, y_te = split_data(texts, labels)
    texts_tr, y_tr = sample_split(texts_tr, y_tr, MAX_TRAIN)
    texts_va, y_va = sample_split(texts_va, y_va, MAX_VAL)
    texts_te, y_te = sample_split(texts_te, y_te, MAX_TEST)
    
    vocab, idf = fit_text_vectorizer(texts_tr, max_features=MAX_FEATURES)
    X_tr = texts_to_tfidf_direct(texts_tr, vocab, idf).T
    X_va = texts_to_tfidf_direct(texts_va, vocab, idf).T
    X_te = texts_to_tfidf_direct(texts_te, vocab, idf).T
    
    # Treinar DNN Melhorada
    dnn = BinaryDNNNumpyImproved(hidden_dims=(256, 128), activation="elu", learning_rate=1e-3, lambda_l2=1e-5, dropout_rate=0.3)
    dnn.fit(X_tr, y_tr, X_va, y_va, epochs=15, batch_size=64, patience=5)
    
    # Avaliação no Test Set
    preds_te = dnn.predict(X_te)
    print_report("DNN TEST SET", y_te, preds_te)
else:
    print("Dataset AI_Human.csv não encontrado. Insira o ficheiro para treinar e avaliar o modelo.")

Total de exemplos: 487231 | [human, ai] = [305797 181434]
Epoch 001 | LR: 0.00100 | train 0.00788 | val 0.02079
Epoch 002 | LR: 0.00100 | train 0.00336 | val 0.02416
Epoch 003 | LR: 0.00100 | train 0.00293 | val 0.02990
Epoch 004 | LR: 0.00090 | train 0.00322 | val 0.03165
Epoch 005 | LR: 0.00081 | train 0.00106 | val 0.03245
Epoch 006 | LR: 0.00073 | train 0.00056 | val 0.02826
Early stopping acionado.
=== DNN TEST SET ===  accuracy=0.9936 | macro_f1=0.9931
  human: precision=0.9936  recall=0.9962  f1=0.9949
     ai: precision=0.9935  recall=0.9893  f1=0.9914
